# 🏥 Clinical ECG Classification: Arrhythmia Detection

**Objective:** Build a recurrent neural network (RNN) capable of identifying abnormal heartbeats in ECG signals.

**The Clinical Stakes:** In a hospital setting, missing a sick patient (False Negative) is far more dangerous than a false alarm (False Positive). Therefore, your primary goal is high **Sensitivity (Recall)**.

**Your Mission:**
The Chief Medical Officer has asked you to evaluate three different sequence models. You must implement, train, and clinically evaluate:
1. `SimpleRNN` (The Baseline)
2. `LSTM` (Long Short-Term Memory)
3. `GRU` (Gated Recurrent Unit)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Import our clinical evaluation metrics from the provided utils.py file
from utils import choose_threshold_by_recall, compute_binary_metrics

# Define global constants
TIMESTEPS = 140
FEATURES = 1
TARGET_RECALL = 0.90
EPOCHS = 30
BATCH_SIZE = 32

## Part 0: Data Loading & Preprocessing
We are using the **ECG5000 dataset**. The data has already been split into Training, Validation, and Testing sets. 

*Note: We only have 400 training samples. You must use regularization to prevent overfitting.*

In [ ]:
def load_and_preprocess_ecg():
    print("Loading ECG5000 Data...")
    # Note for Students: In the actual repository, this will load the .txt files.
    # For this template, we are simulating the exact tensor shapes.
    X_train = np.random.randn(400, TIMESTEPS, FEATURES)
    y_train = np.random.randint(0, 2, 400)
    
    X_val = np.random.randn(100, TIMESTEPS, FEATURES)
    y_val = np.random.randint(0, 2, 100)
    
    X_test = np.random.randn(4500, TIMESTEPS, FEATURES)
    y_test = np.random.randint(0, 2, 4500)
    
    print(f"Train shape: {X_train.shape}")
    print(f"Test shape: {X_test.shape}")
    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = load_and_preprocess_ecg()

## Part 1: The SimpleRNN Baseline
The `SimpleRNN` relies on repeated multiplication over 140 timesteps. This makes it highly susceptible to the **Exploding/Vanishing Gradient Problem**.

**TODO:**
1. Build a `Sequential` model with an `Input` layer, a `SimpleRNN` layer (32 units), and a `Dense` output layer.
2. Compile it using the `Adam` optimizer. 
3. **Crucial:** You must set `clipnorm=1.0` inside your optimizer to prevent exploding gradients (`NaN` loss).

In [ ]:
print("\n--- Building SimpleRNN ---")

# TODO: Define model_rnn
model_rnn = Sequential([
    # YOUR CODE HERE
])

# TODO: Compile model_rnn (Remember clipnorm!)
# YOUR CODE HERE

model_rnn.summary()

# TODO: Train the model using model.fit()
# history_rnn = model_rnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE)

## Part 2: The LSTM (Gradient Highway)
Long Short-Term Memory networks solve the vanishing gradient by adding a **Cell State ($C_t$)**. Because it uses addition instead of multiplication, gradients can flow backward perfectly across all 140 timesteps.

**TODO:**
1. Build an `LSTM` model (64 units).
2. Because LSTMs are mathematically powerful (4 gates), they overfit easily on 400 samples. You **must** add `dropout=0.2` to the LSTM layer.
3. Compile and train.

In [ ]:
print("\n--- Building LSTM ---")

# TODO: Define model_lstm
model_lstm = Sequential([
    # YOUR CODE HERE
])

# TODO: Compile model_lstm
# YOUR CODE HERE

model_lstm.summary()

# TODO: Train the model
# history_lstm = model_lstm.fit(...)

## Part 3: The GRU (The Efficient Alternative)
The Gated Recurrent Unit (GRU) is a modern streamlining of the LSTM. It combines the Forget and Input gates into a single **Update Gate**, resulting in fewer parameters while maintaining excellent long-term memory.

**TODO:**
1. Build a `GRU` model (64 units) with `dropout=0.2`.
2. Compare its parameter count to the LSTM above. It should be noticeably smaller.
3. Compile and train.

In [ ]:
print("\n--- Building GRU ---")

# TODO: Define model_gru
model_gru = Sequential([
    # YOUR CODE HERE
])

# TODO: Compile model_gru
# YOUR CODE HERE

model_gru.summary()

# TODO: Train the model
# history_gru = model_gru.fit(...)

## Part 4: Clinical Evaluation & Threshold Tuning
Accuracy is a deceptive metric. To properly evaluate these models, we must look at **False Negatives**. 
We will evaluate each model twice:
1. **Standard:** Using the default $0.50$ threshold.
2. **Clinical:** Using `choose_threshold_by_recall` to find the safest threshold for a $>90\%$ target.

In [ ]:
def evaluate_model_clinically(model, X_test, y_test, name="Model"):
    # Ensure you have trained the model before calling this!
    try:
        y_prob = model.predict(X_test, verbose=0).ravel()
    except:
        print(f"{name} is not trained yet!")
        return None
        
    # 1. Standard Eval (Threshold 0.5)
    metrics_std, _ = compute_binary_metrics(y_test, y_prob, threshold=0.5)
    
    # 2. Clinical Eval (Tuned for Recall)
    tuned_thresh = choose_threshold_by_recall(y_test, y_prob, target_recall=TARGET_RECALL)
    metrics_tuned, cm = compute_binary_metrics(y_test, y_prob, threshold=tuned_thresh)
    
    print(f"\n{'='*30}")
    print(f"Results for {name}:")
    print(f"{'='*30}")
    print(f"Standard Recall (0.5): {metrics_std['Sensitivity_Recall']:.4f}")
    print(f"Tuned Threshold:       {tuned_thresh:.4f}")
    print(f"Tuned Recall:          {metrics_tuned['Sensitivity_Recall']:.4f}")
    print(f"False Negatives:       {metrics_tuned['FN']} missed patients.")
    
    return metrics_tuned

# TODO: Uncomment these lines after training your models to see the clinical results
# results_rnn = evaluate_model_clinically(model_rnn, X_test, y_test, "SimpleRNN")
# results_lstm = evaluate_model_clinically(model_lstm, X_test, y_test, "LSTM")
# results_gru = evaluate_model_clinically(model_gru, X_test, y_test, "GRU")

## Part 5: Final Analysis
Based on the results generated above, answer the following questions:

### Q1: The Vanishing Gradient
Why did the SimpleRNN struggle to maintain high recall compared to the LSTM/GRU, specifically regarding the 140 timesteps of an ECG signal?

**Your Answer:** 
*(Double click to edit)*

---
### Q2: Parameter Efficiency
Look at the `model.summary()` outputs. Why does the GRU have fewer parameters than the LSTM despite having the same number of hidden units (64)?

**Your Answer:** 
*(Double click to edit)*

---
### Q3: The Clinical Decision
If the LSTM and GRU achieve nearly identical Recall on the test set, but the GRU trains 20% faster, which architecture would you choose to deploy into a wearable heart monitor? Justify your choice.

**Your Answer:** 
*(Double click to edit)*